In [1]:
!pip install ddgs trafilatura -q

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from pprint import pprint
from IPython.display import Markdown, display



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

Key is: sk-proj-


In [3]:

def search_web(query):
    ddgs = DDGS()

    results = ddgs.text(query, max_result = 5)
    pprint(f"Got results: {results}")
    return json.dumps(results, indent=2)

In [4]:
def fetch_url(url: str):

    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(text)
        if text: 
            print(f" \u2705 Got Text")
            return text
    print(f"\u274C fail to fetch text from URL {url}. ")
    return (f"Could not extract text from url {url}. Try different source.")


In [5]:
search_web("AI in healthcare in 2030")

("Got results: [{'title': '7 ways AI is transforming healthcare | World "
 "Economic Forum', 'href': "
 "'https://www.weforum.org/stories/2025/08/ai-transforming-global-health/', "
 "'body': 'With 4.5 billion people currently without access to essential "
 'healthcare services and a health worker shortage of 11 million expected by '
 '2030, AI has the potential to help bridge that gap and revolutionize global '
 "healthcare.'}, {'title': 'AI in Healthcare: 3 Revolutionary Trends by 2030 | "
 "Kindo', 'href': "
 "'https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030', 'body': "
 "'AI transforms healthcare via predictive analytics, precision medicine, and "
 'automated diagnostics. Explore 3 key shifts defining the 2030 medical '
 "landscape.'}, {'title': 'Here are 3 ways AI will change healthcare by 2030 | "
 "World Economic Forum', 'href': "
 "'https://www.weforum.org/stories/2020/01/future-of-artificial-intelligence-healthcare-delivery/', "
 '\'body\': "January 7, 2020 - He

'[\n  {\n    "title": "7 ways AI is transforming healthcare | World Economic Forum",\n    "href": "https://www.weforum.org/stories/2025/08/ai-transforming-global-health/",\n    "body": "With 4.5 billion people currently without access to essential healthcare services and a health worker shortage of 11 million expected by 2030, AI has the potential to help bridge that gap and revolutionize global healthcare."\n  },\n  {\n    "title": "AI in Healthcare: 3 Revolutionary Trends by 2030 | Kindo",\n    "href": "https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030",\n    "body": "AI transforms healthcare via predictive analytics, precision medicine, and automated diagnostics. Explore 3 key shifts defining the 2030 medical landscape."\n  },\n  {\n    "title": "Here are 3 ways AI will change healthcare by 2030 | World Economic Forum",\n    "href": "https://www.weforum.org/stories/2020/01/future-of-artificial-intelligence-healthcare-delivery/",\n    "body": "January 7, 2020 - Healthca

In [6]:
fetch_url("https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f")

AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights
Healthcare is entering a transformative era. The volume of patient data, medical literature, and imaging is growing exponentially, creating new pressures for clinicians and administrative staff. Workforce shortages and increasing patient demand are only amplifying these challenges.
We explored these issues in a recent webinar featuring Jan Beger , a healthcare AI veteran with over 20 years of experience. We discussed how AI can help health systems manage these pressures, streamline workflows, and improve patient outcomes.
In practice, we’ve helped hospitals implement AI tools that prioritize critical imaging, optimize scheduling, and identify high-risk patients for follow-up. These solutions reduce administrative burdens, enhance operational efficiency, and allow clinicians to focus on delivering quality care.
At OSP, we focus on integrating AI seamlessly into workflows, ensuring technology enhances human capabili

"AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights\nHealthcare is entering a transformative era. The volume of patient data, medical literature, and imaging is growing exponentially, creating new pressures for clinicians and administrative staff. Workforce shortages and increasing patient demand are only amplifying these challenges.\nWe explored these issues in a recent webinar featuring Jan Beger , a healthcare AI veteran with over 20 years of experience. We discussed how AI can help health systems manage these pressures, streamline workflows, and improve patient outcomes.\nIn practice, we’ve helped hospitals implement AI tools that prioritize critical imaging, optimize scheduling, and identify high-risk patients for follow-up. These solutions reduce administrative burdens, enhance operational efficiency, and allow clinicians to focus on delivering quality care.\nAt OSP, we focus on integrating AI seamlessly into workflows, ensuring technology enhances human cap

In [7]:
tools = []

search_web_function = {
    "name" : "search_web",
    "description": "search the web using DuckDuckGo and return 3 results",
    "parameters": {
        "type": "object",
        "properties":{
            "query": {
                "type": "string",
                "description": "The search query information on the web"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type":"function", "function": search_web_function} )

#-------------------------------------------

fetch_url_function = {
    "name" : "fetch_url",
    "description": "fetch url the main context from the web page",
    "parameters": {
        "type": "object",
        "properties":{
            "url": {
                "type": "string",
                "description": "fetch url extract text from the web"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type":"function", "function": fetch_url_function} )


In [8]:
tools


[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'search the web using DuckDuckGo and return 3 results',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query information on the web'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'fetch url the main context from the web page',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'fetch url extract text from the web'}},
    'required': ['url']}}}]

In [9]:
def handle_tool_call(tool_calls):
    tools_results = []
    # return what to add to our context (about tool call results, a dictionary)
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        print(f" \U0001f527 Calling function {function_name}")
        args = json.loads(tool_call.function.arguments)

        #Route to function name
        if function_name == "search_web":

            result = search_web(args['query'])
            #print(f"Send notification: {args['message']}")
            content = f"Search Result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args['url'])
            content = f"Fetch result: {result}"
        else:
            content = f"Unknown function {function_name}"

        tool_call_result  = {
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
            
        }

        tools_results.append(tool_call_result)

    return tools_results

## Step 4: System Prompt

In [10]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that
it can be easily parsed and extracted. You CANNOT and SHOULD NOT include "DONE:" 
in any other part of your response except at the very beginning of the final research.
 
Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

## Step 5: Agentic Loop

In [18]:
from json import tool
from urllib import response


def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """ 
    Run a reseach a topic

    Args: 
        topic: The topic to research
        max_iternation: limit to prevent infinite loop
    Return: 
        The research brief

    """

    print("\n \U0001F50D Staring researching on topic: {topic}")
    print("=" * 60)

    messages = [
        {"role":"system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and return a comprehensive research brief: \n\n {topic}"}
    ]

    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        print(f"Iteratin {iteration}")

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools = tools,
            max_tokens=1000
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
        else:
            content = message.content
            if content.startswith("DONE"):
                brief = content[len("DONE:"):].strip()
                print("\n \u2705 Research complete: \n{brief}")

                return brief
            else:

                print(f" \U0001F4AD Agent is thinking: ")
                pprint(content)

        if iteration == max_iterations - 1:
            print(" \u26A0 Stop researching in next iteration")
            messages.append({"role": "user", "content":"YOu have reached max iterations. Please deliver your brief with DONE"})
    
    #fall back return

    return "Research incomplete. Max iterations reached without finalizing brief."



In [12]:
brief = run_research_agent("AI in healthcare in 2030")
display(Markdown(brief))


 🔍 Staring researching on topic: {topic}
Iteratin 1
 🔧 Calling function search_web
("Got results: [{'title': 'AI In Healthcare Market Size & Share | Industry "
 "Report, 2033', 'href': "
 "'https://www.grandviewresearch.com/industry-analysis/artificial-intelligence-ai-healthcare-market', "
 "'body': 'The global regulatory framework for AI in healthcare primarily "
 'regulates artificial intelligence through Software as a Medical Device (SaMD '
 "...'}, {'title': 'AI-Driven Healthcare in 2030 - Insights From Jan Beger "
 "in', 'href': "
 "'https://www.newswire.com/news/ai-driven-healthcare-in-2030-insights-from-jan-beger-in-osp-s-digital-22554546', "
 "'body': 'AI-Driven Healthcare in 2030 ... The webinar has provided some "
 'valuable yet exciting perspectives on what AI-driven healthcare will be in '
 "2030.'}, {'title': 'Artificial Intelligence in Healthcare Market Size to Hit "
 "USD', 'href': "
 "'https://www.precedenceresearch.com/artificial-intelligence-in-healthcare-market', "


Research Brief: AI in Healthcare in 2030

Key Facts and Statistics:
- The global AI in healthcare market was valued at approximately USD 36.67 billion in 2025 and is projected to reach USD 505.59 billion by 2033, growing at a CAGR of 38.9% from 2026 to 2033 (Grand View Research).
- North America accounted for over 54% of the AI healthcare market revenue in 2025, with the U.S. expected to see significant growth due to favorable policies and technological advancements.
- Software solutions dominate the AI healthcare market segment, comprising over 46% of revenue in 2025.
- Robot-assisted surgery was the largest application segment in 2025 with over 13% revenue share.
- Pharmaceutical and biotechnology companies represent the largest end-use segment (over 30% revenue share).
- A Microsoft-IDC 2024 study found 79% of healthcare organizations currently use AI technology, with an ROI seen in just 14 months, generating $3.20 for every dollar invested.

Main Themes and Arguments:

1. Market Growth and Drivers:
- AI adoption in healthcare is driven by the need for improved efficiency, accuracy, and patient outcomes.
- The increasing volume and variety of healthcare data (EHRs, imaging, genomics, wearables) provide opportunities for AI to extract actionable insights and assist clinical decision-making.
- A global shortage of healthcare workers (projected deficit of 10 million by 2033) is accelerating the use of AI to support diagnostics, treatment planning, and workflow automation.
- The COVID-19 pandemic expedited AI deployment especially in diagnostics, claims processing, and cybersecurity.

2. Future of Care - Proactive, Personalized, Autonomous:
- AI will shift healthcare from episodic, reactive care to continuous, proactive, and hyper-personalized medicine.
- Predictive analytics and wearable data will enable early disease detection and prevention before clinical symptoms onset.
- Development of “digital twins” will simulate individual responses to treatments for optimized interventions.
- Autonomous AI agents will independently manage routine clinical tasks such as triage, history synthesis, and even treatment initiation, especially in low resource or emergency scenarios.
- Ethical governance and regulatory frameworks (e.g., SaMD regulations, IMDRF Good Machine Learning Practices) are evolving to ensure AI accountability and safety.

3. Technological Innovations and Applications:
- Generative AI (Large Language Models, GANs) herald a new era for drug design, medical education, and patient engagement with personalized, context-aware content and simulations.
- AI-powered robotics and digital co-pilots will enhance clinical workflows, automated documentation, and patient monitoring—especially for elderly and chronically ill patients.
- The convergence of AI with multi-omics data (genomics, proteomics, metabolomics) will underpin hyper-personalized treatments, creating individual health profiles to better predict drug and therapy responses.
- Machine learning remains the dominant technology for extracting insights from complex healthcare datasets, particularly for diagnostics and treatment planning.
- AI will increasingly optimize hospital operations by automating admissions, scheduling, billing, resource management, and compliance monitoring, leading to cost savings and better patient experiences.

4. Regional and Industry Insights:
- North America leads the market with advanced infrastructure, investment, and regulatory support.
- Asia Pacific and Europe are fast-growing markets due to innovation, government initiatives, and private sector investments.
- Leading companies include Microsoft, IBM, NVIDIA, Intel, GE Healthcare, Oracle, Medtronic, Merck, and IQVIA.
- Recent innovations include AI-driven stroke detection in hospitals, AI medical scribes, virtual care platforms, and AI-powered drug discovery partnerships.

Notable Data Points:
- 38.9% CAGR growth for AI in healthcare from 2026-2033.
- AI adoption by 79% of healthcare organizations (2024).
- AI reduces drug discovery timelines from 5-6 years to about one year.
- AI-assisted robot surgery and fraud detection fastest growing sub-segments.
- AI returns $3.20 per $1 invested, typically within 14 months.
- Projected global shortage of 10 million healthcare workers by 2033 pushes AI adoption.

Summary:
By 2030, AI will fundamentally transform healthcare delivery and hospital management from reactive to predictive and personalized systems. Advanced AI technologies—including machine learning, generative AI, and context-aware computing—will improve diagnostics, drug discovery, robotic surgery, and patient engagement. Autonomous AI agents and digital twins will enable novel interventions tailored to individual biology and lifestyles, moving care into continuous health management. Operationally, AI will automate administrative tasks, optimize resource use and compliance, resulting in safer, more efficient, and cost-effective healthcare delivery globally. Regulatory standards and ethical governance will be critical to ensure safe, equitable use of AI in medicine. The AI healthcare ecosystem will be driven by major tech and healthcare players optimizing through partnerships, innovations, and

In [20]:
JUDGE_PROMPT = """# Judge Prompt: Insufficient Research Breadth (TRUE/FALSE)

You are scoring a research agent's deliverable research brief to determine 
if there was a
research breadth failure. Return only TRUE or FALSE.

Definitions

Source:
Any distinctly identifiable origin of information referenced in the brief. This
includes named websites, publications, reports, databases, articles, papers, 
organizations, or any other distinguishable information source. Multiple references to the same 
source count as one source.

Research breadth failure (label TRUE):
The research brief references fewer than 6
distinct sources. When counting:

1. Distinct sources only:
If the same website, publication, or organization is referenced
multiple times, it counts as one source.

2. Identifiable references required:
A source must be specific enough to be
independently locatable — vague attributions like "studies show," "experts say," or
"according to reports" do NOT count as sources.

3. Embedded or inline references count:
Sources do not need to appear in a formal
bibliography. References woven into the body text (e.g., "according to a McKinsey 2024
report" or "data from the Bureau of Labor Statistics") count as valid sources.

4. Agent's own reasoning is not a source:
The agent's synthesis, analysis, or opinions do
not count toward the source total.

No research breadth failure (label FALSE):
The research brief references 6 or more
distinct, identifiable sources anywhere in the deliverable — whether in a bibliography,
inline citations, or body-text attributions.

Output Format

Return exactly one token: TRUE or FALSE. No explanations.

"""

In [21]:
TOPICS = [
    "The impact of generative AI on pharmaceutical drug discovery"
    "Comparative analysis of nuclear fusion startups and their commercial viability"
    "How central bank digital currencies are reshaping cross-border payments"
    "The state of microplastics regulation in the European Union"
    "Remote work's long-term effects on commercial real estate valuations"
    "Advances in solid-state battery technology for electric vehicles"
    "The role of satellite internet constellations in bridging the global digital divide"
    "Youth mental health trends and social media platform accountability"
    "Water scarcity solutions in the Middle East: desalination vs. conservation"
    "The economic impact of sports stadium subsidies on local communities"
]

In [22]:
JUDGE_MODEL = "gpt-4.1"
MODEL = "gpt-4.1-nano"


In [24]:
import contextlib
import io


results = []

for i, topic in enumerate(TOPICS):
    # 1. Run agent
    # suppress printout from the agent for cleaner eval logs  
    with contextlib.redirect_stdout(io.StringIO()):
        brief = run_research_agent(topic, max_iterations=10)

    # 2. Judge
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": brief},
        ],
        max_tokens=1,
        temperature=0,
    )
    verdict = response.choices[0].message.content.strip()
    failed = verdict == "TRUE"

    results.append({"topic": topic, "verdict": verdict, "failed": failed})
    print(f"  → {'FAIL' if failed else 'PASS'}")
    print(brief)
# --- Summary ---
n_fail = sum(r["failed"] for r in results)
print(f"\n{'='*50}")
print(f"Results: {len(results) - n_fail}/{len(results)} passed  |  {n_fail}/{len(results)} failed")
print(f"{'='*50}")
for r in results:
    status = "FAIL ✗" if r["failed"] else "PASS ✓"
    print(f"  [{status}] {r['topic'][:65]}")

  → PASS
Research Brief on Multiple Topics

1. The Impact of Generative AI on Pharmaceutical Drug Discovery
- Generative AI (GAI), leveraging deep learning, large language models (LLMs), and neural networks such as VAEs, GANs, transformers, and reinforcement learning, is revolutionizing drug discovery by enabling de novo drug design—creating novel molecules designed for desired pharmacokinetic and pharmacodynamic properties.
- AI accelerates target identification, molecular property prediction, compound generation, synthesis planning, virtual screening, and clinical trial success forecasting.
- Significant models include Insilico Medicine’s Pharma.AI, NVIDIA’s BioNeMo, and OpenAI’s ChatGPT-based tools like DrugGPT.
- Benefits: Speeding up drug development, cost reduction (Insilico reported completing preclinical to Phase 1 trials in 2.5 years and 1/10th the cost versus traditional methods), generating unique molecules, aiding in interpreting complex biological data.
- Challenges: Immen